In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ── 1. 加载数据 ──────────────────────────────────────────
ratings = pd.read_csv("../data/ml-1m/ratings.dat", sep="::",
    names=["user_id","movie_id","rating","timestamp"], engine="python")

# ── 2. ID 重映射（从0开始，方便 Embedding 索引）────────────
# 原始 user_id: 1~6040，movie_id: 1~3952（不连续）
user2idx = {u: i for i, u in enumerate(ratings["user_id"].unique())}
item2idx = {m: i for i, m in enumerate(ratings["movie_id"].unique())}

ratings["uid"] = ratings["user_id"].map(user2idx)
ratings["iid"] = ratings["movie_id"].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)
print(f"用户数: {n_users}, 物品数: {n_items}")

# ── 3. 按时间切分（和 Week 7 一样，防止 data leakage）────────
ratings_sorted = ratings.sort_values("timestamp")
split = int(len(ratings_sorted) * 0.9)
train_df = ratings_sorted.iloc[:split]
val_df   = ratings_sorted.iloc[split:]

# ── 4. 构建"用户看过的物品"集合（用于负采样时排除）─────────
user_pos_items = train_df.groupby("uid")["iid"].apply(set).to_dict()
print(f"训练集: {len(train_df)} 条，验证集: {len(val_df)} 条")

# ── 5. 负采样 Dataset ───────────────────────────────────
class NCFDataset(Dataset):
    """
    每条正样本 (user, item, label=1) 配 neg_ratio 条负样本 (user, rand_item, label=0)
    负样本从用户未交互过的物品中随机采
    """
    def __init__(self, df, user_pos_items, n_items, neg_ratio=4):
        self.users, self.items, self.labels = [], [], []
        
        for _, row in df.iterrows():
            uid, iid = int(row["uid"]), int(row["iid"])
            # 正样本
            self.users.append(uid)
            self.items.append(iid)
            self.labels.append(1.0)
            # 负样本
            pos_set = user_pos_items.get(uid, set())
            sampled = 0
            while sampled < neg_ratio:
                neg = np.random.randint(0, n_items)
                if neg not in pos_set:
                    self.users.append(uid)
                    self.items.append(neg)
                    self.labels.append(0.0)
                    sampled += 1
        
        self.users  = torch.tensor(self.users,  dtype=torch.long)
        self.items  = torch.tensor(self.items,  dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.float)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

# 先用小 subset 测试速度（训练集 100k 条，避免等太久）
train_sub = train_df.sample(n=100_000, random_state=42)
train_dataset = NCFDataset(train_sub, user_pos_items, n_items, neg_ratio=4)
print(f"Dataset 大小: {len(train_dataset)} 条（正+负）")
print(f"正负比例 = 1 : 4，正样本 {len(train_sub)}, 负样本 {len(train_sub)*4}")

# 检查一条样本
u, i, l = train_dataset[0]
print(f"\n样本示例 → uid={u.item()}, iid={i.item()}, label={l.item()}")

用户数: 6040, 物品数: 3706
训练集: 900188 条，验证集: 100021 条
Dataset 大小: 500000 条（正+负）
正负比例 = 1 : 4，正样本 100000, 负样本 400000

样本示例 → uid=4264, iid=46, label=1.0


In [2]:
import torch.nn as nn
import torch.nn.functional as F

class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, mlp_dims=[64, 32, 16]):
        super().__init__()
        
        # MF 路径的 embedding
        self.mf_user_emb = nn.Embedding(n_users, emb_dim)
        self.mf_item_emb = nn.Embedding(n_items, emb_dim)
        
        # MLP 路径的 embedding（独立的，不共享）
        self.mlp_user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp_item_emb = nn.Embedding(n_items, emb_dim)
        
        # MLP 层：输入是两个 emb_dim 拼接 = 2*emb_dim
        mlp_layers = []
        in_dim = emb_dim * 2
        for out_dim in mlp_dims:
            mlp_layers += [nn.Linear(in_dim, out_dim), nn.ReLU()]
            in_dim = out_dim
        self.mlp = nn.Sequential(*mlp_layers)
        
        # 最终输出层：MF输出(emb_dim) + MLP最后一层输出(mlp_dims[-1])
        self.output_layer = nn.Linear(emb_dim + mlp_dims[-1], 1)
        
        self._init_weights()
    
    def _init_weights(self):
        for emb in [self.mf_user_emb, self.mf_item_emb,
                    self.mlp_user_emb, self.mlp_item_emb]:
            nn.init.normal_(emb.weight, std=0.01)
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
        nn.init.kaiming_uniform_(self.output_layer.weight)
    
    def forward(self, user_ids, item_ids):
        # MF 路径：element-wise 乘积
        mf_u = self.mf_user_emb(user_ids)
        mf_i = self.mf_item_emb(item_ids)
        mf_out = mf_u * mf_i                        # [B, emb_dim]
        
        # MLP 路径：拼接后过 MLP
        mlp_u = self.mlp_user_emb(user_ids)
        mlp_i = self.mlp_item_emb(item_ids)
        mlp_out = self.mlp(torch.cat([mlp_u, mlp_i], dim=1))  # [B, 16]
        
        # 合并两路
        combined = torch.cat([mf_out, mlp_out], dim=1)        # [B, emb_dim+16]
        logit = self.output_layer(combined).squeeze(1)         # [B]
        return torch.sigmoid(logit)

# 实例化并检查结构
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = NCF(n_users, n_items, emb_dim=32, mlp_dims=[64, 32, 16]).to(device)
print(model)
print(f"\n设备: {device}")
print(f"可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 验证 forward pass（随机输入）
dummy_u = torch.randint(0, n_users, (4,)).to(device)
dummy_i = torch.randint(0, n_items, (4,)).to(device)
dummy_out = model(dummy_u, dummy_i)
print(f"\nforward 验证 → 输出 shape: {dummy_out.shape}, 值域: [{dummy_out.min().item():.3f}, {dummy_out.max().item():.3f}]")

NCF(
  (mf_user_emb): Embedding(6040, 32)
  (mf_item_emb): Embedding(3706, 32)
  (mlp_user_emb): Embedding(6040, 32)
  (mlp_item_emb): Embedding(3706, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
  )
  (output_layer): Linear(in_features=48, out_features=1, bias=True)
)

设备: mps
可训练参数: 630,561

forward 验证 → 输出 shape: torch.Size([4]), 值域: [0.531, 0.533]


In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for users, items, labels in loader:
        users  = users.to(device)
        items  = items.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        preds = model(users, items)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

# 先跑 1 个 epoch 看 loss 是否在下降
print("训练中...")
loss = train_epoch(model, train_loader, optimizer, criterion, device)
print(f"Epoch 1 → train loss: {loss:.4f}")

训练中...
Epoch 1 → train loss: 0.4159


In [4]:
def evaluate(model, val_df, user_pos_items, n_items, device, K=10, n_neg=99, n_users_eval=1000):
    model.eval()
    hits, ndcgs = [], []
    
    # 只评估部分用户（加速）
    eval_users = val_df["uid"].unique()
    if len(eval_users) > n_users_eval:
        eval_users = np.random.choice(eval_users, n_users_eval, replace=False)
    
    with torch.no_grad():
        for uid in eval_users:
            # 该用户在验证集里的正样本
            pos_items = val_df[val_df["uid"] == uid]["iid"].values
            if len(pos_items) == 0:
                continue
            pos_item = pos_items[0]  # 取第一条
            
            # 采 99 个负样本（用户没交互过的）
            all_pos = user_pos_items.get(uid, set())
            negs = []
            while len(negs) < n_neg:
                neg = np.random.randint(0, n_items)
                if neg not in all_pos:
                    negs.append(neg)
            
            # 100个候选：1正 + 99负
            candidates = [pos_item] + negs
            u_tensor = torch.tensor([uid] * 100, dtype=torch.long).to(device)
            i_tensor = torch.tensor(candidates,  dtype=torch.long).to(device)
            
            scores = model(u_tensor, i_tensor).cpu().numpy()
            
            # 按分数排序，找正样本的排名（0-indexed）
            rank = np.argsort(-scores).tolist().index(0)  # 正样本在 candidates[0]
            
            hits.append(1 if rank < K else 0)
            ndcgs.append(1 / np.log2(rank + 2) if rank < K else 0)
    
    return np.mean(hits), np.mean(ndcgs)

# 评估当前模型（只训练了1个epoch）
hr, ndcg = evaluate(model, val_df, user_pos_items, n_items, device)
print(f"Epoch 1 → HR@10: {hr:.4f}, NDCG@10: {ndcg:.4f}")
print(f"\n随机基准（理论值）：HR@10 = {10/100:.2f}, NDCG@10 ≈ 0.063")

Epoch 1 → HR@10: 0.4680, NDCG@10: 0.2746

随机基准（理论值）：HR@10 = 0.10, NDCG@10 ≈ 0.063


In [5]:
best_hr = 0
best_epoch = 0
history = []

for epoch in range(1, 11):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    hr, ndcg = evaluate(model, val_df, user_pos_items, n_items, device)
    history.append({"epoch": epoch, "loss": loss, "hr": hr, "ndcg": ndcg})
    print(f"Epoch {epoch:2d} → loss: {loss:.4f} | HR@10: {hr:.4f} | NDCG@10: {ndcg:.4f}")
    
    if hr > best_hr:
        best_hr = hr
        best_epoch = epoch
        torch.save(model.state_dict(), "../data/ncf_best.pth")

print(f"\n最佳模型：Epoch {best_epoch}, HR@10: {best_hr:.4f}")

Epoch  1 → loss: 0.3492 | HR@10: 0.4920 | NDCG@10: 0.2826
Epoch  2 → loss: 0.3039 | HR@10: 0.4970 | NDCG@10: 0.2910
Epoch  3 → loss: 0.2499 | HR@10: 0.4890 | NDCG@10: 0.2789
Epoch  4 → loss: 0.2039 | HR@10: 0.4740 | NDCG@10: 0.2697
Epoch  5 → loss: 0.1673 | HR@10: 0.4570 | NDCG@10: 0.2559
Epoch  6 → loss: 0.1378 | HR@10: 0.4650 | NDCG@10: 0.2515
Epoch  7 → loss: 0.1137 | HR@10: 0.4520 | NDCG@10: 0.2481
Epoch  8 → loss: 0.0936 | HR@10: 0.4450 | NDCG@10: 0.2425
Epoch  9 → loss: 0.0769 | HR@10: 0.4240 | NDCG@10: 0.2308
Epoch 10 → loss: 0.0626 | HR@10: 0.4240 | NDCG@10: 0.2264

最佳模型：Epoch 2, HR@10: 0.4970


In [6]:
# 全量训练集 + 加 Dropout + weight decay
class NCFWithDropout(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, mlp_dims=[64, 32, 16], dropout=0.2):
        super().__init__()
        self.mf_user_emb = nn.Embedding(n_users, emb_dim)
        self.mf_item_emb = nn.Embedding(n_items, emb_dim)
        self.mlp_user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp_item_emb = nn.Embedding(n_items, emb_dim)
        
        mlp_layers = []
        in_dim = emb_dim * 2
        for out_dim in mlp_dims:
            mlp_layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*mlp_layers)
        
        self.output_layer = nn.Linear(emb_dim + mlp_dims[-1], 1)
        self._init_weights()
    
    def _init_weights(self):
        for emb in [self.mf_user_emb, self.mf_item_emb,
                    self.mlp_user_emb, self.mlp_item_emb]:
            nn.init.normal_(emb.weight, std=0.01)
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
        nn.init.kaiming_uniform_(self.output_layer.weight)
    
    def forward(self, user_ids, item_ids):
        mf_u  = self.mf_user_emb(user_ids)
        mf_i  = self.mf_item_emb(item_ids)
        mf_out = mf_u * mf_i

        mlp_u  = self.mlp_user_emb(user_ids)
        mlp_i  = self.mlp_item_emb(item_ids)
        mlp_out = self.mlp(torch.cat([mlp_u, mlp_i], dim=1))

        combined = torch.cat([mf_out, mlp_out], dim=1)
        return torch.sigmoid(self.output_layer(combined).squeeze(1))

# 全量数据集（会花几分钟建 Dataset）
print("构建全量 Dataset...")
full_train_dataset = NCFDataset(train_df, user_pos_items, n_items, neg_ratio=4)
full_train_loader  = DataLoader(full_train_dataset, batch_size=2048, shuffle=True, num_workers=0)
print(f"全量 Dataset 大小: {len(full_train_dataset):,} 条")

# 新模型 + weight_decay
model2 = NCFWithDropout(n_users, n_items, emb_dim=32, mlp_dims=[64, 32, 16], dropout=0.2).to(device)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3, weight_decay=1e-5)

best_hr2, best_epoch2 = 0, 0

for epoch in range(1, 11):
    loss = train_epoch(model2, full_train_loader, optimizer2, criterion, device)
    hr, ndcg = evaluate(model2, val_df, user_pos_items, n_items, device)
    print(f"Epoch {epoch:2d} → loss: {loss:.4f} | HR@10: {hr:.4f} | NDCG@10: {ndcg:.4f}")
    
    if hr > best_hr2:
        best_hr2 = hr
        best_epoch2 = epoch
        torch.save(model2.state_dict(), "../data/ncf_best_v2.pth")

print(f"\n最佳模型：Epoch {best_epoch2}, HR@10: {best_hr2:.4f}")
print(f"对比 v1 最佳：HR@10: {best_hr:.4f} (Epoch {best_epoch})")

构建全量 Dataset...
全量 Dataset 大小: 4,500,940 条
Epoch  1 → loss: 0.3686 | HR@10: 0.4930 | NDCG@10: 0.2832
Epoch  2 → loss: 0.3354 | HR@10: 0.5040 | NDCG@10: 0.2924
Epoch  3 → loss: 0.3109 | HR@10: 0.5360 | NDCG@10: 0.3179
Epoch  4 → loss: 0.2974 | HR@10: 0.5440 | NDCG@10: 0.3244
Epoch  5 → loss: 0.2887 | HR@10: 0.5140 | NDCG@10: 0.2990
Epoch  6 → loss: 0.2829 | HR@10: 0.5450 | NDCG@10: 0.3126
Epoch  7 → loss: 0.2784 | HR@10: 0.5470 | NDCG@10: 0.3160
Epoch  8 → loss: 0.2748 | HR@10: 0.5390 | NDCG@10: 0.3177
Epoch  9 → loss: 0.2719 | HR@10: 0.5490 | NDCG@10: 0.3154
Epoch 10 → loss: 0.2695 | HR@10: 0.5440 | NDCG@10: 0.3148

最佳模型：Epoch 9, HR@10: 0.5490
对比 v1 最佳：HR@10: 0.4970 (Epoch 2)


In [9]:
print("构建全量 Dataset...")
full_train_dataset2 = NCFDataset(train_df, user_pos_items, n_items, neg_ratio=2)
full_train_loader2  = DataLoader(full_train_dataset2, batch_size=1024, shuffle=True, num_workers=0)
print(f"全量 Dataset 大小: {len(full_train_dataset2):,} 条")

# 新模型 + weight_decay
model3 = NCFWithDropout(n_users, n_items, emb_dim=64, mlp_dims=[64, 32, 16], dropout=0.1).to(device)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3, weight_decay=1e-5)

best_hr3, best_epoch3 = 0, 0

for epoch in range(1, 11):
    loss = train_epoch(model3, full_train_loader2, optimizer3, criterion, device)
    hr, ndcg = evaluate(model3, val_df, user_pos_items, n_items, device)
    print(f"Epoch {epoch:2d} → loss: {loss:.4f} | HR@10: {hr:.4f} | NDCG@10: {ndcg:.4f}")
    
    if hr > best_hr3:
        best_hr3 = hr
        best_epoch3 = epoch
        torch.save(model3.state_dict(), "../data/ncf_best_v3.pth")

print(f"\n最佳模型：Epoch {best_epoch3}, HR@10: {best_hr3:.4f}")
print(f"对比 v2 最佳：HR@10: {best_hr2:.4f} (Epoch {best_epoch2})")

构建全量 Dataset...
全量 Dataset 大小: 2,700,564 条
Epoch  1 → loss: 0.4475 | HR@10: 0.5030 | NDCG@10: 0.2901
Epoch  2 → loss: 0.3996 | HR@10: 0.5180 | NDCG@10: 0.2954
Epoch  3 → loss: 0.3714 | HR@10: 0.5410 | NDCG@10: 0.3105
Epoch  4 → loss: 0.3524 | HR@10: 0.5490 | NDCG@10: 0.3095
Epoch  5 → loss: 0.3410 | HR@10: 0.5560 | NDCG@10: 0.3146
Epoch  6 → loss: 0.3324 | HR@10: 0.5630 | NDCG@10: 0.3237
Epoch  7 → loss: 0.3251 | HR@10: 0.5340 | NDCG@10: 0.3008
Epoch  8 → loss: 0.3195 | HR@10: 0.5430 | NDCG@10: 0.3047
Epoch  9 → loss: 0.3150 | HR@10: 0.5500 | NDCG@10: 0.3163
Epoch 10 → loss: 0.3106 | HR@10: 0.5440 | NDCG@10: 0.3187

最佳模型：Epoch 6, HR@10: 0.5630
对比 v2 最佳：HR@10: 0.5490 (Epoch 9)
